In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math, copy, time, collections

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.autograd import Variable

In [ ]:
import sys
import os

# Go up one level to the project root (add more '..' if you are deeper)
project_root = os.path.abspath('..')
print(f"This is the project_root{project_root}")
if project_root not in sys.path:
    sys.path.append(project_root)
!git clone https://github.com/tom-esplin/midi-generator-model

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on {}".format(device))

Running on cpu


In [3]:
import sys
import os

# Go up one level to the project root (add more '..' if you are deeper)
project_root = os.path.abspath('..')
print(f"This is the project_root{project_root}")
if project_root not in sys.path:
    sys.path.append(project_root)
!git clone https://github.com/tom-esplin/midi-generator-model

This is the project_rootc:\Users\Archibald-RA\Documents\music-generator-project


fatal: destination path 'midi-generator-model' already exists and is not an empty directory.


In [10]:
genre = "folk"
from training.prep_dataset import MidiDataset, ContinuousMidiDataset
from pathlib import Path
from miditok import PerTok,TokSequence
#add function to autoselect genre path
#start with small dataset
CHUNK_SIZE = 1000
exp_path = Path("tokenization","saved_tokens","folk-0-04-04-2026_12-22-01")
tokenizer = PerTok(params=Path(exp_path,"tokenizer.json"))
print(f"Loaded Tokenizer Vocab Size: {len(tokenizer)}")
def create_mask(sequence_length):
    t = torch.full((sequence_length,sequence_length),-torch.inf)
    return torch.triu(t,1).to(device)

Loaded Tokenizer Vocab Size: 30000


In [4]:
@torch.no_grad()
def generate(x, model, pred_length, temperature=.8):
    """
    Generate `pred_length` characters, given the starting sequence x.
    The model's output can be turned into a character prediction using
    softmax (divide the logits by temperature first), and then sampled
    from the multinomial distribution.
    Return the tensor of sampled integers.

    Make sure to provide the correct-shaped mask each iteration.
    """
    for i in range(pred_length):
      mask = create_mask(x.shape[0])
      output = model(x,mask) / temperature
      preds = torch.softmax(output,dim=-1)
      last_pred = preds[:,-1]
      x = torch.cat([x,torch.multinomial(last_pred,1)],dim=1)
    return x

In [17]:
eval_song = "001013_0.mid"
EVAL_START_TOKEN_IDS = torch.tensor(tokenizer(Path("prepared_data","folk","test",eval_song))[0].ids).unsqueeze(0)
EVAL_START_TOKEN_IDS = EVAL_START_TOKEN_IDS[:,:CHUNK_SIZE].to(device)
def evaluate(model,iteration_count):
    """Runs generate function and formats and prints predictions.
    """
    model.eval()
    inference_batch = generate(EVAL_START_TOKEN_IDS, model, CHUNK_SIZE)
    for i,inference_tokens in enumerate(inference_batch):
        list_of_ids = inference_tokens.squeeze().tolist()
        tok_sequence = TokSequence(ids=list_of_ids,are_ids_encoded=True)
        pred_midi = tokenizer.decode([tok_sequence])
        pred_midi.dump_midi(Path(f"test_{iteration_count}.mid"))
    model.train()

In [18]:
evaluate(small_model,0)
print(f"tokenize vocab size: {tokenizer.vocab_size}")

tokenize vocab size: 30000


In [6]:
import time
from datetime import datetime
from tqdm import tqdm

def train(model, optimizer, dataloader, n_optimization_steps, eval_interval):
    # Label smoothing adds a bit of uncertainty to the model, improving text generation.
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    losses = []
    mask = create_mask(CHUNK_SIZE)
    model.train()
    start_time = time.time()
    data_iter = iter(dataloader)
    for step in tqdm(range(n_optimization_steps)):
      try:
        batch = next(data_iter)
        x = batch[:,:-1]
        y_truth = batch[:,1:]
      except StopIteration as e:
        data_iter = iter(dataloader)
        batch = next(data_iter)
        x = batch[:,:-1]
        y_truth = batch[:,1:]
      x = x.to(device)
      y_truth = y_truth.to(device)
      optimizer.zero_grad()
      y_hat = model(x,mask)
      loss = loss_fn(torch.transpose(y_hat,1,2),y_truth)
      loss.backward()
      optimizer.step()
      losses.append(loss.item())
      if (step + 1) % eval_interval == 0:
        model.eval()
        print(f"\nStep:{step} Loss {losses[-1]}\n\n")
        evaluate(model,step)
        model.train()
      if time.time() - start_time >= 360000:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        torch.save(model.state_dict(),'model_{}_{}'.format(timestamp, step))
        torch.save(optimizer.state_dict(),'optimizer_{}_{}'.format(timestamp, step))


    return losses

In [7]:



from models.transformer import TransformerDecoder
from torch.optim import Adam

dataset = MidiDataset(folder_path=Path(exp_path,"test"),chunk_size=CHUNK_SIZE)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

small_model = TransformerDecoder(tokenizer.vocab_size, N=2, d_model=256, d_ff=512, h=2).to(device)

optimizer = Adam(small_model.parameters(), lr=5e-4)

losses = train(small_model, optimizer, dataloader, n_optimization_steps=2000, eval_interval=1000)

  0%|          | 7/2000 [02:14<10:38:16, 19.22s/it]


KeyboardInterrupt: 

In [19]:
losses = train(small_model, optimizer, dataloader, n_optimization_steps=2000, eval_interval=100)

  5%|▍         | 99/2000 [31:29<9:02:39, 17.13s/it] 


Step:99 Loss 7.60021448135376




 10%|▉         | 199/2000 [1:03:24<8:46:11, 17.53s/it]


Step:199 Loss 7.859979629516602




 15%|█▍        | 299/2000 [1:35:14<8:25:56, 17.85s/it] 

...

 95%|█████████▍| 1899/2000 [10:01:20<30:23, 18.06s/it] 


Step:1899 Loss 6.449914455413818




100%|█████████▉| 1999/2000 [10:32:55<00:18, 18.10s/it]  


Step:1999 Loss 6.643756866455078




100%|██████████| 2000/2000 [10:35:00<00:00, 19.05s/it]


In [ ]:
# generate some music